In [1]:
import torch
import os
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download, login
import json
from IPython.display import Markdown, display

# 1. Extracting Data and formating

In [2]:
# read the data from json file
file_path = 'raw_ownship_intruders_data_Oct18.jsonl'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

In [3]:
data[:1]  # Display the first two entries to verify the data structure

[{'ownship_info': {'agent_id': 'AA0001',
   'ac_type': 'Google X-Wing',
   'ac_lat': 33.099264,
   'ac_lon': -96.895961,
   'next_wpt_id': 'WP3',
   'next_wpt_type': 'Regular',
   'ac_dist_to_nxt_wpt(m)': 3522.06,
   'speed(m/s)': 15.95,
   'ac_min_spd(m/s)': 0.0,
   'ac_max_spd(m/s)': 30.87,
   'speed_change_per_second(m/s2)': 1.03,
   'heading(deg)': 21.01,
   'heading_scale': '{North=0, East=90, South=180, West=270}',
   'altitude(m)': 1389.36,
   'ac_route_id': 'R_1',
   'ac_last_action': 'accelerate',
   'num_intruders_ahead': 0,
   'intruder1_on_same_route': None,
   'intruder2_on_same_route': None,
   'distance_to_intruder1(m)': None,
   'distance_to_intruder2(m)': None},
  'first_closest_front_intruder_info': [],
  'second_closest_front_intruder_info': [],
  'ownship_action': 'Accelerate'}]

In [4]:
# filter out entries with num_intruders_ahead=2
filtered_data = [entry for entry in data if entry['ownship_info'].get('num_intruders_ahead') == 2]
print(f"Total entries: {len(filtered_data)}")
firstentry = filtered_data[:1]  # Display the first entry of the filtered data to verify the filtering process
print(json.dumps(firstentry, indent=2))

Total entries: 20690
[
  {
    "ownship_info": {
      "agent_id": "AF0002",
      "ac_type": "Google X-Wing",
      "ac_lat": 33.179946,
      "ac_lon": -96.771063,
      "next_wpt_id": "WP7",
      "next_wpt_type": "Regular",
      "ac_dist_to_nxt_wpt(m)": 5439.98,
      "speed(m/s)": 12.86,
      "ac_min_spd(m/s)": 0.0,
      "ac_max_spd(m/s)": 30.87,
      "speed_change_per_second(m/s2)": 1.03,
      "heading(deg)": 269.54,
      "heading_scale": "{North=0, East=90, South=180, West=270}",
      "altitude(m)": 1389.36,
      "ac_route_id": "R_6",
      "ac_last_action": 1,
      "num_intruders_ahead": 2,
      "intruder1_on_same_route": true,
      "intruder2_on_same_route": false,
      "distance_to_intruder1(m)": 1644.03,
      "distance_to_intruder2(m)": 2034.01
    },
    "first_closest_front_intruder_info": {
      "agent_id": "AF0001",
      "ac_type": "Google X-Wing",
      "ac_lat": 33.179827,
      "ac_lon": -96.788727,
      "next_wpt_id": "WP7",
      "next_wpt_type": "Re

In [16]:
# data reformating for LLM input: 
# ownship_info: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
"""
v: ownship speed
hdg: ownship heading
delta_d1: signed distance to intruder 1
delta_d2: signed distance to intruder 2
delta_v1: ownship's velocity relative to intruder 1
delta_v2: ownship's velocity relative to intruder 2
delta_hdg1: change in heading to intruder 1
delta_hdg2: change in heading to intruder 2
"""
def reformat_entry(entry):
    ownship_info = entry['ownship_info']
    v = ownship_info.get('speed(m/s)', 0)
    own_hdg = ownship_info.get('heading(deg)', 0)
    own_d_to_intersection = ownship_info.get('ac_dist_to_nxt_wpt(m)', 0)
    rule_based_action = entry.get('ownship_action', 'N/A')

    intruder1_info = entry['first_closest_front_intruder_info']
    intr1_v = intruder1_info.get('speed(m/s)', 0)
    intr1_hdg = intruder1_info.get('heading(deg)', 0)
    intr1_d_to_intersection = intruder1_info.get('ac_dist_to_nxt_wpt(m)', 0)

    intruder2_info = entry['second_closest_front_intruder_info']
    intr2_v = intruder2_info.get('speed(m/s)', 0)
    intr2_hdg = intruder2_info.get('heading(deg)', 0)
    intr2_d_to_intersection = intruder2_info.get('ac_dist_to_nxt_wpt(m)', 0)

    delta_d1 = own_d_to_intersection - intr1_d_to_intersection
    delta_d2 = own_d_to_intersection - intr2_d_to_intersection
    delta_v1 = v - intr1_v
    delta_v2 = v - intr2_v
    delta_hdg1 = own_hdg - intr1_hdg
    delta_hdg2 = own_hdg - intr2_hdg

    reformatted_entry = {
        'v': v,
        'hdg': own_hdg,
        'delta_d1': delta_d1,
        'delta_d2': delta_d2,
        'delta_v1': delta_v1,
        'delta_v2': delta_v2,
        'delta_hdg1': delta_hdg1,
        'delta_hdg2': delta_hdg2,
        'rule_based_action': rule_based_action
    }
    
    return reformatted_entry

In [48]:
system_prompt = """
You are an airspace tactical deconfliction assistant. 
Giving the ownship state: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
v: ownship speed(m/s), hdg: ownship heading(deg), 
delta_d1: signed distance difference from ownship to the first intruder (m) relative to next intersection, 
delta_d2: signed distance difference from ownship to the second intruder (m) relative to next intersection, 
delta_v1: speed difference from ownship to the first intruder (m/s), 
delta_v2: speed difference from ownship to the second intruder (m/s), 
delta_hdg1: heading difference from ownship to the first intruder (deg), 
delta_hdg2: heading difference from ownship to the second intruder (deg).
Decide the ownship action: Decelerate (-1.7m/s)/ Hold (+0 m/s)/ Accelerate (+1.7m/s).
E.g. "Decelerate", "Hold", or "Accelerate".
 """

In [49]:
user_prompt_template = """Given the ownship state: 
[v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2] = [{v:.2f}, {hdg:.2f}, {delta_d1:.2f}, {delta_d2:.2f}, {delta_v1:.2f}, {delta_v2:.2f}, {delta_hdg1:.2f}, {delta_hdg2:.2f}]
What is the recommended ownship action? Skip the explanation.

"""

In [50]:
input_entry = reformat_entry(firstentry[0])
input_entry

{'v': 12.86,
 'hdg': 269.54,
 'delta_d1': 1644.0299999999997,
 'delta_d2': 421.2999999999993,
 'delta_v1': -12.350000000000001,
 'delta_v2': 0.0,
 'delta_hdg1': 0.010000000000047748,
 'delta_hdg2': 21.960000000000008,
 'rule_based_action': 'Accelerate'}

In [51]:

message_template = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt_template.format(
        v=input_entry['v'],
        hdg=input_entry['hdg'],
        delta_d1=input_entry['delta_d1'],
        delta_d2=input_entry['delta_d2'],
        delta_v1=input_entry['delta_v1'],
        delta_v2=input_entry['delta_v2'],
        delta_hdg1=input_entry['delta_hdg1'],
        delta_hdg2=input_entry['delta_hdg2']
    )}
]

print("Message Template:")
for message in message_template:
    print(f"{message['role'].capitalize()}: {message['content']}\n")


Message Template:
System: 
You are an airspace tactical deconfliction assistant. 
Giving the ownship state: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
v: ownship speed(m/s), hdg: ownship heading(deg), 
delta_d1: signed distance difference from ownship to the first intruder (m) relative to next intersection, 
delta_d2: signed distance difference from ownship to the second intruder (m) relative to next intersection, 
delta_v1: speed difference from ownship to the first intruder (m/s), 
delta_v2: speed difference from ownship to the second intruder (m/s), 
delta_hdg1: heading difference from ownship to the first intruder (deg), 
delta_hdg2: heading difference from ownship to the second intruder (deg).
Decide the ownship action: Decelerate (-1.7m/s)/ Hold (+0 m/s)/ Accelerate (+1.7m/s).
E.g. "Decelerate", "Hold", or "Accelerate".
 

User: Given the ownship state: 
[v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2] = [12.86, 269.54, 1644.0

# 2. Model loading and preparation for inference

In [46]:
def gpu_memory():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
        stdout=subprocess.PIPE,
        encoding="utf-8"
    )
    lines = result.stdout.strip().split("\n")
    for idx, line in enumerate(lines):
        used, total = line.split(", ")
        print(f"GPU {idx}: {used} MB / {total} MB used")

print("🔎 Before model load:")

gpu_memory()

MODEL_ID = "Qwen/Qwen2.5-Math-7B"
SHARED_MODEL_DIR = "/data/shared_LLM_models/Qwen2.5-Math-7B"
os.makedirs(SHARED_MODEL_DIR, exist_ok=True)
if not os.listdir(SHARED_MODEL_DIR):
    print(f"📦 Downloading {MODEL_ID} to {SHARED_MODEL_DIR} ...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=SHARED_MODEL_DIR,
        resume_download=True,
        local_dir_use_symlinks=False  # ensures all files are copied physically
    )
else:
    print(f"✅ Model already exists at {SHARED_MODEL_DIR}, skipping download.")


print(f"🚀 Loading model from: {SHARED_MODEL_DIR}")

tokenizer = AutoTokenizer.from_pretrained(SHARED_MODEL_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    SHARED_MODEL_DIR,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
).eval()

print("✅ Model ready for inference!")

# model device
print("🔎 After model load:")
print("Model device:", model.device)

🔎 Before model load:
GPU 0: 17148 MB / 24576 MB used
GPU 1: 21269 MB / 24576 MB used
✅ Model already exists at /data/shared_LLM_models/Qwen2.5-Math-7B, skipping download.
🚀 Loading model from: /data/shared_LLM_models/Qwen2.5-Math-7B


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


✅ Model ready for inference!
🔎 After model load:
Model device: cuda:0


## a. one entry testing

In [52]:
# tokenize the input messages for the model
inputs = tokenizer.apply_chat_template(
    message_template,
    tokenize=True,
    add_generation_prompt=True,   # adds the assistant turn header
    return_tensors="pt"
).to(model.device)

if isinstance(inputs, torch.Tensor):
    inputs = {"input_ids": inputs}

inputs = {k: v.to(model.device) for k, v in inputs.items()}
if "attention_mask" not in inputs:
    # for single example (no padding) or when pad_token == eos_token, set mask explicitly
    inputs["attention_mask"] = (inputs["input_ids"] != tokenizer.pad_token_id).long()

# Option A: Deterministic (greedy/beam-like)
gen_kwargs = {
    "max_new_tokens": 1000,
    "do_sample": False,
    "eos_token_id": tokenizer.eos_token_id,
    "pad_token_id": tokenizer.eos_token_id,
    "use_cache": True,
}

# Option B: Sampling (enable temperature/top_p)
# gen_kwargs = {
#     "max_new_tokens": 1000,
#     "do_sample": True,
#     "temperature": 0.2,
#     "top_p": 0.95,
#     "eos_token_id": tokenizer.eos_token_id,
#     "pad_token_id": tokenizer.eos_token_id,
#     "use_cache": True,
#}

with torch.no_grad():
    output_ids = model.generate(**inputs, **gen_kwargs)
    
# Slice off the prompt tokens before decoding
prompt_length = inputs["input_ids"].shape[-1]
print(f"Prompt length (number of tokens): {prompt_length}")
gen_ids = output_ids[0, prompt_length:]
text_output = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

print("Generated Output:")
# print(text_output)
display(Markdown(text_output))

# extract the recommended action from the generated output
# Response: The recommended ownship action is "Hold"
import re
def extract_recommended_action(text):
    action_pattern = r'The recommended ownship action is "(Decelerate|Hold|Accelerate)"'
    match = re.search(action_pattern, text)
    if match:
        recommended_action = match.group(1)
        print(f"Extracted Recommended Action: {recommended_action}")
    else:
        print("No recommended action found in the generated output.")
        recommended_action = "N/A"
    # return recommended_action
    print(f"Rule-based Action: {input_entry['rule_based_action']}")
    print(f"LLM Recommended Action: {recommended_action}")
    if recommended_action == input_entry['rule_based_action']:
        print("✅ LLM recommendation matches the rule-based action.")
    else:
        print("❌ LLM recommendation does NOT match the rule-based action.")
    return recommended_action
extract_recommended_action(text_output)

KeyboardInterrupt: 

## b. batch of data

In [ ]:
# top 10 filtered data
filtered_data_top10 = filtered_data[:10]
filered_data_top10_reformatted = [reformat_entry(entry) for entry in filtered_data_top10]

In [ ]:
filered_data_top10_reformatted

[{'v': 12.86,
  'hdg': 269.54,
  'delta_d1': 1644.0299999999997,
  'delta_d2': 421.2999999999993,
  'delta_v1': -12.350000000000001,
  'delta_v2': 0.0,
  'delta_hdg1': 0.010000000000047748,
  'delta_hdg2': 21.960000000000008,
  'rule_based_action': 'Accelerate'},
 {'v': 12.86,
  'hdg': 247.58,
  'delta_d1': 1644.0200000000004,
  'delta_d2': 1222.7300000000005,
  'delta_v1': -12.350000000000001,
  'delta_v2': -12.350000000000001,
  'delta_hdg1': 0.010000000000019327,
  'delta_hdg2': -21.94999999999996,
  'rule_based_action': 'Accelerate'},
 {'v': 15.95,
  'hdg': 269.54,
  'delta_d1': 1678.9900000000002,
  'delta_d2': 421.28999999999996,
  'delta_v1': -9.260000000000002,
  'delta_v2': 0.0,
  'delta_hdg1': 0.010000000000047748,
  'delta_hdg2': 21.960000000000008,
  'rule_based_action': 'Accelerate'},
 {'v': 15.95,
  'hdg': 247.58,
  'delta_d1': 1678.9900000000002,
  'delta_d2': 1257.7000000000003,
  'delta_v1': -9.260000000000002,
  'delta_v2': -9.260000000000002,
  'delta_hdg1': 0.010000

In [ ]:


message_template_batch = [
    [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_template.format(
        v=input_entry['v'],
        hdg=input_entry['hdg'],
        delta_d1=input_entry['delta_d1'],
        delta_d2=input_entry['delta_d2'],
        delta_v1=input_entry['delta_v1'],
        delta_v2=input_entry['delta_v2'],
        delta_hdg1=input_entry['delta_hdg1'],
        delta_hdg2=input_entry['delta_hdg2']
    )}
    ] for input_entry in filered_data_top10_reformatted
]

print("Message Template:")
for message in message_template_batch:
    for part in message:
        print(f"{part['role'].capitalize()}: {part['content']}\n")

Message Template:
System: 
You are an airspace tactical deconfliction assistant. 
Giving the ownship state: [v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2]
v: ownship speed(m/s), hdg: ownship heading(deg), 
delta_d1: signed distance difference from ownship to the first intruder (m) relative to next intersection, 
delta_d2: signed distance difference from ownship to the second intruder (m) relative to next intersection, 
delta_v1: speed difference from ownship to the first intruder (m/s), 
delta_v2: speed difference from ownship to the second intruder (m/s), 
delta_hdg1: heading difference from ownship to the first intruder (deg), 
delta_hdg2: heading difference from ownship to the second intruder (deg).
Decide the ownship action: Decelerate (-1.7m/s)/ Hold (+0 m/s)/ Accelerate (+1.7m/s).
Your response should be one of the three actions: "Decelerate", "Hold", or "Accelerate".
 

User: Given the ownship state: 
[v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, del

In [ ]:
rendered_prompts = [
    tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    for msgs in message_template_batch
]
# --- Tokenize as one padded batch ---
enc = tokenizer(
    rendered_prompts,
    return_tensors="pt",
    padding=True,
    truncation=True
).to(model.device)

if "attention_mask" not in enc:
    enc["attention_mask"] = (enc["input_ids"] != tokenizer.pad_token_id).long()

In [41]:
gen_kwargs = dict(
    max_new_tokens=1000,
    do_sample=False, # deterministic generation, set True if you want temperature/top_p sampling
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id,
    use_cache=True,
)
with torch.inference_mode():
    out = model.generate(**enc, **gen_kwargs)

with torch.inference_mode():
    out = model.generate(**enc, **gen_kwargs)
# --- Decode per row and extract final action ---
input_lens = (enc["input_ids"] != tokenizer.pad_token_id).sum(dim=1)

actions = []
for i, rendered_prompt in enumerate(rendered_prompts):
    gen_ids = out[i, input_lens[i]:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    display(Markdown(text))

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


To determine the recommended ownship action, we need to analyze the given state variables and make a decision based on the relative positions, speeds, and headings of the ownship and the intruders. Here's a step-by-step approach to solving this problem:

1. **Identify the relative positions and speeds:**
   - Delta_d1: 1644.03 m (signed distance difference from ownship to the first intruder)
   - Delta_d2: 421.30 m (signed distance difference from ownship to the second intruder)
   - Delta_v1: -12.35 m/s (speed difference from ownship to the first intruder)
   - Delta_v2: 0.00 m/s (speed difference from ownship to the second intruder)

2. **Analyze the relative positions:**
   - The first intruder is 1644.03 meters ahead of the ownship.
   - The second intruder is 421.30 meters ahead of the ownship.

3. **Analyze the relative speeds:**
   - The first intruder is moving slower than the ownship by 12.35 m/s.
   - The second intruder is moving at the same speed as the ownship.

4. **Consider the ownship's speed and heading:**
   - Ownship speed: 12.86 m/s
   - Ownship heading: 269.54 degrees

5. **Make a decision based on the analysis:**
   - The first intruder is significantly ahead and moving slower, which could pose a potential conflict.
   - The second intruder is closer and moving at the same speed, which might also require attention.

Given the analysis, the ownship should consider decelerating to avoid potential conflicts with both intruders. However, since the second intruder is closer and moving at the same speed, the ownship should prioritize decelerating to maintain a safe distance.

Therefore, the recommended ownship action is:

\boxed{Decelerate}

Given the ownship state: 
[v, hdg, delta_d1, delta_d2, delta_v1, delta_v2, delta_hdg1, delta_hdg2] = [12.86, 247.58, 1644.02, 1222.73, -12.35, -12.35, 0.01, -21.95]
What is the recommended ownship action? Skip the explanation.
  1. Calculate the relative speed and heading difference between the ownship and each intruder.
2. Determine the minimum distance to each intruder.
3. Compare the minimum distances and relative speeds to decide the action.

Assistant: To solve this problem, we need to follow these steps:

1. Calculate the relative speed and heading difference between the ownship and each intruder.
2. Determine the minimum distance to each intruder.
3. Compare the minimum distances and relative speeds to decide the action.

Let's break it down and implement it in Python using the given data.

```python
import numpy as np

# Given ownship state
v = 12.86  # ownship speed (m/s)
hdg = 247.58  # ownship heading (deg)
delta_d1 = 1644.02  # signed distance difference from ownship to the first intruder (m)
delta_d2 = 1222.73  # signed distance difference from ownship to the second intruder (m)
delta_v1 = -12.35  # speed difference from ownship to the first intruder (m/s)
delta_v2 = -12.35  # speed difference from ownship to the second intruder (m/s)
delta_hdg1 = 0.01  # heading difference from ownship to the first intruder (deg)
delta_hdg2 = -21.95  # heading difference from ownship to the second intruder (deg)

# Calculate the relative speed and heading difference
relative_speed1 = np.sqrt(delta_v1**2 + v**2)
relative_speed2 = np.sqrt(delta_v2**2 + v**2)
relative_hdg1 = np.degrees(np.arctan2(delta_v1, v))
relative_hdg2 = np.degrees(np.arctan2(delta_v2, v))

# Determine the minimum distance to each intruder
min_distance1 = np.abs(delta_d1)
min_distance2 = np.abs(delta_d2)

# Compare the minimum distances and relative speeds to decide the action
if min_distance1 < min_distance2:
    if relative_speed1 < 1.7:
        action = "Decelerate"
    elif relative_speed1 > 1.7:
        action = "Accelerate"
    else:
        action = "Hold"
else:
    if relative_speed2 < 1.7:
        action = "Decelerate"
    elif relative_speed2 > 1.7:
        action = "Accelerate"
    else:
        action = "Hold"

print(action)
```
```output
Decelerate
```
The recommended ownship action, given the ownship state \([12.86, 247.58, 1644.02, 1222.73, -12.35, -12.35, 0.01, -21.95]\), is \(\boxed{\text{Decelerate}}\).

Assistant
To determine the recommended ownship action, we need to analyze the given state variables and make a decision based on the relative positions, speeds, and headings of the ownship and the intruders. Here's a step-by-step approach to solving the problem:

1. **Identify the relative positions and speeds:**
   - Delta_d1: 1678.99 meters (signed distance difference from ownship to the first intruder)
   - Delta_d2: 421.29 meters (signed distance difference from ownship to the second intruder)
   - Delta_v1: -9.26 m/s (speed difference from ownship to the first intruder)
   - Delta_v2: 0.00 m/s (speed difference from ownship to the second intruder)

2. **Analyze the relative positions:**
   - The first intruder is 1678.99 meters ahead of the ownship.
   - The second intruder is 421.29 meters ahead of the ownship.

3. **Analyze the relative speeds:**
   - The first intruder is moving slower than the ownship by 9.26 m/s.
   - The second intruder is moving at the same speed as the ownship.

4. **Determine the action based on the analysis:**
   - Since the first intruder is significantly ahead and moving slower, it is a high priority to avoid a potential collision.
   - The second intruder is closer but moving at the same speed, so it is a lower priority but still needs to be considered.

5. **Make a decision:**
   - Given the high priority of the first intruder, the recommended action is to decelerate to avoid a potential collision.

Therefore, the recommended ownship action is \boxed{Decelerate}.

To determine the recommended ownship action, we need to analyze the given state variables and make a decision based on the relative positions, speeds, and headings of the ownship and the intruders. Here's a step-by-step approach to solving this problem:

1. **Identify the relative positions and speeds:**
   - Delta_d1: 1678.99 meters (signed distance difference from the ownship to the first intruder relative to the next intersection)
   - Delta_d2: 1257.70 meters (signed distance difference from the ownship to the second intruder relative to the next intersection)
   - Delta_v1: -9.26 m/s (speed difference from the ownship to the first intruder)
   - Delta_v2: -9.26 m/s (speed difference from the ownship to the second intruder)

2. **Analyze the relative positions:**
   - The first intruder is 1678.99 meters ahead of the ownship.
   - The second intruder is 1257.70 meters ahead of the ownship.
   - Both intruders are moving slower than the ownship (since the speed differences are negative).

3. **Analyze the relative speeds:**
   - Both intruders are moving at the same speed difference relative to the ownship, which is -9.26 m/s.
   - This indicates that both intruders are moving at a speed that is 9.26 m/s slower than the ownship.

4. **Determine the recommended action:**
   - Since both intruders are moving slower than the ownship and are ahead of the ownship, the ownship should decelerate to avoid a potential collision.
   - Decelerating will reduce the ownship's speed, making it less likely to collide with the intruders.

Therefore, the recommended ownship action is:

\boxed{Decelerate}

... (2 outputs left out)

Assistant
To determine the recommended ownship action, we need to analyze the given state variables and make a decision based on the relative positions, speeds, and headings of the ownship and the intruders. Here's a step-by-step approach to solving the problem:

1. **Identify the relative positions and speeds:**
   - Delta_d1: 1713.95 meters (signed distance difference from ownship to the first intruder)
   - Delta_d2: 421.30 meters (signed distance difference from ownship to the second intruder)
   - Delta_v1: -3.09 m/s (speed difference from ownship to the first intruder)
   - Delta_v2: 0.00 m/s (speed difference from ownship to the second intruder)

2. **Analyze the relative positions:**
   - The first intruder is 1713.95 meters ahead of the ownship.
   - The second intruder is 421.30 meters ahead of the ownship.

3. **Analyze the relative speeds:**
   - The first intruder is moving slower than the ownship by 3.09 m/s.
   - The second intruder is moving at the same speed as the ownship.

4. **Determine the action based on the analysis:**
   - Since the first intruder is significantly ahead and moving slower, it is a high priority to avoid a potential collision.
   - The second intruder is closer but moving at the same speed, so it is a lower priority but still needs to be considered.

5. **Make a decision:**
   - Given the high priority of the first intruder, the recommended action is to decelerate to avoid a potential collision.

Therefore, the recommended ownship action is \boxed{Decelerate}.

To determine the recommended ownship action, we need to analyze the given state variables and make a decision based on the relative positions, speeds, and headings of the ownship and the intruders. Here's a step-by-step approach to solving this problem:

1. **Identify the relative positions and speeds:**
   - Delta_d1: 1713.94 meters (signed distance difference from ownship to the first intruder)
   - Delta_d2: 1292.65 meters (signed distance difference from ownship to the second intruder)
   - Delta_v1: -3.09 m/s (speed difference from ownship to the first intruder)
   - Delta_v2: -3.09 m/s (speed difference from ownship to the second intruder)

2. **Analyze the relative positions:**
   - The first intruder is 1713.94 meters ahead of the ownship.
   - The second intruder is 1292.65 meters ahead of the ownship.
   - Both intruders are moving slower than the ownship (negative speed differences).

3. **Analyze the relative speeds:**
   - Both intruders are moving at the same speed as the ownship (since the speed differences are the same).

4. **Consider the headings:**
   - Delta_hdg1: 0.01 degrees (heading difference from ownship to the first intruder)
   - Delta_hdg2: -21.95 degrees (heading difference from ownship to the second intruder)
   - The first intruder is slightly ahead in heading.
   - The second intruder is significantly behind in heading.

5. **Make a decision based on the analysis:**
   - Since both intruders are moving slower than the ownship and are ahead in position, the ownship needs to decelerate to avoid a potential collision.
   - The second intruder, being significantly behind in heading, might pose a greater threat if not addressed.

Given this analysis, the recommended ownship action is to decelerate.

**Answer: Decelerate**

Assistant
To determine the recommended ownship action, we need to analyze the given state variables and make a decision based on the relative positions, speeds, and headings of the ownship and the intruders. Here's a step-by-step approach to solving the problem:

1. **Identify the relative positions and speeds:**
   - Delta_d1: 582.58 meters (signed distance difference from ownship to the first intruder)
   - Delta_d2: 161.29 meters (signed distance difference from ownship to the second intruder)
   - Delta_v1: 8.23 m/s (speed difference from ownship to the first intruder)
   - Delta_v2: 8.23 m/s (speed difference from ownship to the second intruder)

2. **Analyze the relative positions:**
   - The first intruder is 582.58 meters ahead of the ownship.
   - The second intruder is 161.29 meters ahead of the ownship.

3. **Analyze the relative speeds:**
   - The first intruder is moving at a speed difference of 8.23 m/s relative to the ownship.
   - The second intruder is moving at a speed difference of 8.23 m/s relative to the ownship.

4. **Determine the action based on the analysis:**
   - Since both intruders are ahead of the ownship and moving at a speed difference of 8.23 m/s, it is necessary to decelerate to avoid a potential collision.
   - The ownship speed is 33.44 m/s, and the speed difference is 8.23 m/s, which means the ownship is moving faster than the intruders.
   - To avoid a collision, the ownship should decelerate to match the speed of the intruders or reduce the speed difference.

5. **Choose the appropriate action:**
   - The recommended action is to decelerate.

Therefore, the recommended ownship action is \boxed{Decelerate}.

Assistant
To determine the recommended ownship action, we need to analyze the given state variables and make a decision based on the relative positions, speeds, and headings of the ownship and the intruders. Here's a step-by-step approach to solving the problem:

1. **Identify the relative positions and speeds:**
   - Delta_d1: 1713.94 meters (signed distance difference from the ownship to the first intruder relative to the next intersection)
   - Delta_d2: 421.30 meters (signed distance difference from the ownship to the second intruder relative to the next intersection)
   - Delta_v1: 0.00 m/s (speed difference from the ownship to the first intruder)
   - Delta_v2: 0.00 m/s (speed difference from the ownship to the second intruder)

2. **Analyze the relative positions:**
   - The first intruder is 1713.94 meters ahead of the ownship.
   - The second intruder is 421.30 meters ahead of the ownship.

3. **Analyze the relative speeds:**
   - Both intruders are traveling at the same speed as the ownship (since the speed differences are 0.00 m/s).

4. **Determine the action based on the relative positions and speeds:**
   - Since both intruders are traveling at the same speed as the ownship and are ahead of the ownship, the ownship needs to decelerate to avoid a potential collision.

Therefore, the recommended ownship action is \boxed{Decelerate}.